# Gator Quant Hacks: 8-K Option Strategy Research

Canonical submission notebook. The default path uses only committed aggregate Markdown and makes no API requests. The optional judge path is disabled until explicitly enabled, requires the authentic freeze guard and a private Massive API key, and evaluates only the fixed F1 specification for the requested dates. This notebook does not run historical experiments, modify the ledger, or open judge-sealed observations.

## 1. Research Question

The pre-registered hypothesis was that a fresh leadership-change 8-K signals an adverse governance shock that the option market overprices; a post-filing cash-secured put should therefore outperform ordinary days and stale filings. The measured strategy instead underperformed ordinary days in both historically reported windows. That rejects the pre-registered directional claim. The later interpretation that the option market may underprice these shocks is exploratory and unconfirmed; the negative short-put result does not establish a profitable protective-put strategy.

The event family is five leadership tags: CEO/CFO appointment or departure and executive-officer appointment. The frozen freshness proxy uses EDGAR `CONFORMED PERIOD OF REPORT` relative to the after-close-shifted filing session; fresh is lag at most one trading session. This date is not independently validated as the first public disclosure.

## 2. Configuration

Only dates are configurable in the optional judge path. Strategy, horizons, neighboring parameters, costs, and seeds are displayed and checked against the recorded F1 specification. June–August 2023 in the starter notebook is a configured HOLDOUT placeholder, not a verified date range for judges' sealed data.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
from submission_pipeline import (
    BUILD_COMMIT, FIXED_HORIZONS, PRIMARY, committed_summaries,
    markdown_tables, sensitivity_rows, validate_configuration,
    run_judge, load_metrics, SubmissionBlocked,
)

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"
RUN_CUSTOM_JUDGE = False
AUTHORIZE_RESTRICTED_DATES = False
ENTRY = "post"
HORIZONS = [1, 2, 3, 5, 10, 21, 42, 63]
OTM_GRID = [0.03, 0.05, 0.10]
EXPIRY_BUCKETS = {"1m": (21, 45, 30), "2m": (46, 80, 60), "3-6m": (90, 180, 120)}
COST_HAIRCUT = 0.05
BOOTSTRAP_SEED = 0
DIFFERENCE_SEED = 1
validate_configuration(ENTRY, HORIZONS, OTM_GRID, EXPIRY_BUCKETS,
                       COST_HAIRCUT, BOOTSTRAP_SEED, DIFFERENCE_SEED)
JUDGE_RESULT = None
print(f"Pinned evidence build: {BUILD_COMMIT}; custom judge execution: {RUN_CUSTOM_JUDGE}")

## 3. Environment and Massive API

Default execution imports only the offline summary reader, pandas, and plotting support. It needs no key, does not initialize a network session, and reads a closed allowlist of committed aggregate Markdown using `git show` at the pinned build commit. The optional path loads `MASSIVE_API_KEY` from the environment or repository `.env` without displaying it, uses the original notebook API wrappers, and caches locally. Before loading research definitions it checks the original starter notebook, `pair_test.py`, and `pairings.json` byte-for-byte against the pinned build; it also invokes the original freeze guard. Missing provenance stops with an explicit recovery diagnostic. The original engine calendar bounds supported event and pricing dates.

## 4. Research Journey

This is a static record of meaningful prior tests. Historical experiments are not rerun. `n` is reported in the underlying summary and can differ by estimand; unavailable denominators remain marked unavailable. Experiments 8–12 and 9B are absent from the published snapshot available at this build. They are not classified or assigned results here.

In [ ]:
SUMMARY = committed_summaries()
HISTORY = pd.DataFrame([
    {"Experiment": "Excerpt-date gate", "Hypothesis / strategy": "Use excerpt dates for leadership freshness", "Stage": "Measurement gate", "N": "218; 102 undated", "Classification": "Measurement failure", "Lesson": "46.8% undated; gate stopped before this design's P&L."},
    {"Experiment": "P leadership-low", "Hypothesis / strategy": "Low-JEV leadership / cash-secured put", "Stage": "In-sample", "N": "150 priced", "Classification": "Null", "Lesson": "Reported edge −0.53%; no supported effect."},
    {"Experiment": "A1 CEO-change-low", "Hypothesis / strategy": "Low-JEV CEO change / put and covered call", "Stage": "In-sample", "N": "31", "Classification": "Null / fragile", "Lesson": "Reported −1.01% put; −1.78% covered call."},
    {"Experiment": "A3 CFO/executive-low", "Hypothesis / strategy": "Low-JEV CFO/executive appointment / put and covered call", "Stage": "In-sample", "N": "115–116", "Classification": "Null", "Lesson": "Reported −0.58% put; −1.10% covered call."},
    {"Experiment": "A2 abrupt CEO exit", "Hypothesis / strategy": "High-JEV exit / protective put", "Stage": "Counts only", "N": "3", "Classification": "Feasibility / sparse", "Lesson": "Demoted A→B; no pricing or economic conclusion."},
    {"Experiment": "A4 restructuring high", "Hypothesis / strategy": "High-JEV restructuring / collar", "Stage": "Counts only", "N": "0", "Classification": "Feasibility / sparse", "Lesson": "Demoted A→B; no pricing or economic conclusion."},
    {"Experiment": "B1 cybersecurity low", "Hypothesis / strategy": "Low-JEV cybersecurity / CSP", "Stage": "Counts only", "N": "0", "Classification": "Feasibility failure", "Lesson": "Demoted B→C; no economic test."},
    {"Experiment": "B2 cybersecurity high", "Hypothesis / strategy": "High-JEV breach / protective put", "Stage": "Counts only", "N": "4", "Classification": "Feasibility failure", "Lesson": "Demoted B→C; no economic test."},
    {"Experiment": "B3 director departure high", "Hypothesis / strategy": "High-JEV departure / protective put", "Stage": "Counts only", "N": "16", "Classification": "Feasibility failure", "Lesson": "Demoted B→C; no economic test."},
    {"Experiment": "B4 strategic initiative high", "Hypothesis / strategy": "High-JEV strategic event / long call and collar", "Stage": "Counts only", "N": "0", "Classification": "Feasibility failure", "Lesson": "Demoted B→C; JEV does not encode direction."},
    {"Experiment": "B5 good-news call", "Hypothesis / strategy": "Good-news tags / long call", "Stage": "In-sample", "N": "103 priced", "Classification": "Null", "Lesson": "Reported edge −0.15%; tags included adverse language cases."},
    {"Experiment": "Exploratory business-line exit", "Hypothesis / strategy": "Business-line exit / collar", "Stage": "Small exploratory IS; OOS gate unavailable", "N": "11 IS; 2 reported OOS events", "Classification": "Dirty exploratory / validation failure", "Lesson": "+1.03% at 0/3 significant horizons; repeats, recaps and earnings mentions contaminated events."},
    {"Experiment": "B6 earnings", "Hypothesis / strategy": "Earnings / put and covered call", "Stage": "IS then reported OOS", "N": "117–118 IS; OOS n unavailable", "Classification": "Failed OOS", "Lesson": "+0.39% put and +0.83% call IS; −0.46% / −1.59% reported OOS."},
    {"Experiment": "F1 fresh leadership", "Hypothesis / strategy": "Fresh leadership / post-close 5% OTM 3–6m cash-secured put", "Stage": "IS and reported 2026 OOS", "N": "55 IS; 25 OOS", "Classification": "Pre-registered claim rejected", "Lesson": "Fresh-vs-ordinary edges −1.19% IS and −0.70% OOS; CIs not fully reported."},
    {"Experiment": "VRP 18-category map", "Hypothesis / strategy": "Event-implied move gap; descriptive put mapping", "Stage": "IS map", "N": "18 categories", "Classification": "Null map", "Lesson": "Zero BH passes at q=.10; every reported interval includes zero."},
    {"Experiment": "VRP H2 pooled", "Hypothesis / strategy": "Fresh minus stale outside F1 tags", "Stage": "IS and reported OOS", "N": "666/547 IS; 229/218 OOS", "Classification": "Not confirmed", "Lesson": "+0.003 IS; −0.149 OOS; neither supports the proposed generalization."},
    {"Experiment": "VRP counts repair", "Hypothesis / strategy": "Build category event universe", "Stage": "Counts implementation", "N": "investment_impairment OOS reported 0 after repair", "Classification": "Implementation/data-integrity failure", "Lesson": "Missing tickers field was repaired upstream; not a market result; not independently rerun."},
    {"Experiment": "Experiments 8–12/9B", "Hypothesis / strategy": "Not in this published snapshot", "Stage": "Unavailable", "N": None, "Classification": "Missing / unmerged", "Lesson": "No result or classification inferred."},
])
display(HISTORY)


In [ ]:
# Static experiment-funnel view; no historical stage is executed here.
fig, ax = plt.subplots(figsize=(11, 2.5))
ax.axis("off")
steps = [(0.02, "Signal measurement\nexcerpt dates failed"), (0.27, "Early pairings\nnull / sparse"),
         (0.52, "F1 frozen test\nclaim rejected"), (0.77, "VRP map\n18 tests, 0 BH passes")]
for i, (x, label) in enumerate(steps):
    ax.text(x, .5, label, transform=ax.transAxes, ha="left", va="center",
            bbox={"boxstyle":"round,pad=.6", "fc":"#eaf1f8", "ec":"#32506e"}, fontsize=10)
    if i < len(steps)-1:
        ax.annotate("", xy=(steps[i+1][0]-.025, .5), xytext=(x+.19, .5),
                    xycoords=ax.transAxes, textcoords=ax.transAxes,
                    arrowprops={"arrowstyle":"->", "lw":1.5, "color":"#52677b"})
ax.set_title("Research path from measurement gate to aggregate findings", loc="left")
plt.show()

## 5. Methodological Evolution

The early leadership pairings used excerpt text and JEV/System One. The reported latest internal check was .80 balanced accuracy on 69 labels, with 0% controls scored high. The scorer measures abrupt/planned character, not direction, magnitude, or numeric guidance. These are project gates, not organizer scores or evidence of blind validity. The excerpt-date measurement gate failed before this F1 economic design because many excerpts had no usable event date. F1 instead used EDGAR period-of-report dates and fixed a fresh/stale lag rule before its reported P&L. That rule is a proxy, not proof of first public disclosure. Historical freeze tags are referenced by the reports but are absent from the available checkout, so the optional judge path stops until authentic provenance is restored.

The F1 outcome was not used to choose the frozen primary. Ordinary-day controls sample the same names at least 30 days from tagged events. F1 intervals use an independent row bootstrap (2,000 draws, seed 1 for differences), not issuer-cluster inference. The separate VRP map used 18 category tests with Benjamini–Hochberg correction at q=.10 and month-resampled intervals. No category passed. Its pooled H2 test is separate from the category map.

## 6. Final Signal Construction

For an explicitly enabled judge run, the original harness applies the five frozen leadership tags, EDGAR date proxy, after-close shift, one event per ticker/session, and fresh/stale split. This notebook calls that existing code and does not duplicate or tune signal logic. The custom path is disabled by default. June–August 2023 is only the starter notebook's configured holdout placeholder; the judges' actual sealed dates are unknown here.

In [ ]:
if RUN_CUSTOM_JUDGE:
    JUDGE_RESULT = run_judge(
        START_DATE, END_DATE, enabled=True,
        authorize_restricted_dates=AUTHORIZE_RESTRICTED_DATES,
    )
    print({k: JUDGE_RESULT[k] for k in (
        "start", "end", "n_signal", "n_stale", "issuers_in_event_pool",
        "priced_event_bucket_pairs", "ordinary_draws", "drop_reasons")})
else:
    print("Offline presentation mode: no signal generation, key loading, cache setup, or HTTP.")

## 7. Signal Validation

The committed F1 aggregate reports 62 fresh and 108 stale in-sample events, of which 55 and 95 respectively were priced for the headline comparison. The reported 2026 OOS pool had 27 fresh and 42 stale events; 25 and 40 were priced. These are aggregate counts. The reported human audit found 10/10 reviewed fresh/stale assignments consistent with the period-of-report proxy; this is not an independent blind review and does not establish earliest public disclosure. No independent blind semantic review is available in this snapshot.
Reported discovered, priced, and auxiliary denominators are different stages or estimands; the committed sources do not reconcile them row by row. The table below keeps them separate.

In [ ]:
stage_counts = pd.DataFrame([
    {"window":"in-sample", "stage":"events discovered", "fresh":62, "stale":108},
    {"window":"in-sample", "stage":"priced headline comparison", "fresh":55, "stale":95},
    {"window":"reported 2026 OOS", "stage":"events discovered", "fresh":27, "stale":42},
    {"window":"reported 2026 OOS", "stage":"priced headline comparison", "fresh":25, "stale":40},
    {"window":"in-sample", "stage":"capacity sample (not primary cohort)", "fresh":56, "stale":None},
    {"window":"reported 2026 OOS", "stage":"capacity sample", "fresh":25, "stale":None},
    {"window":"in-sample", "stage":"calendar-year edges (not primary cohort)", "fresh":"27 (2024) + 32 (2025)", "stale":None},
    {"window":"reported 2026 OOS", "stage":"calendar-year edges (not primary cohort)", "fresh":27, "stale":None},
])
display(stage_counts)


## 8. Trade Construction

The primary instrument is a cash-secured put, entered at the post-filing session close, strike 5% below spot, in the 3–6 month expiry bucket (90–180 calendar days; 120-day target). Returns are marked at sessions 21, 42, and expiry for the reported headline. The separate portfolio summary holds for 21 sessions with one tenth capital per event. Transaction cost is modeled as 5% of option premium per side; quotes are unavailable. Ordinary-day controls use the same names. Contract liquidity is thin; corporate-action adjustments, dividends, and early assignment are limitations. The strategy does not match the observed negative edge as a trade recommendation.

## 9. Primary Result

The pre-registered primary contrast is fresh minus stale; the headline also reports fresh versus ordinary days. The negative headline edge is the event-minus-ordinary gross strategy edge averaged over h=21, h=42, and expiry. The reported sample size is the maximum valid per-horizon event count, not a verified identical matched set. Exact F1 confidence interval endpoints, issuer counts, and absolute signal/control means are unavailable in the committed summaries. CI inclusion status is retained exactly; no interval endpoints or net edge are reconstructed.

In [ ]:
F1_RESULTS = markdown_tables(SUMMARY["runs/FINDINGS.md"])[0]
display(pd.DataFrame(F1_RESULTS))
# Point estimates only. CI endpoint values are unavailable in the committed summary.
def signed_number(value):
    return float(value.replace("−", "-").replace("+", "").replace("%", "").strip())
plot_rows = [r for r in F1_RESULTS if r.get("Edge", "").strip()]
fig, ax = plt.subplots(figsize=(8, 3.5))
labels = [f"{r['Window']}: {r['Comparison']} (n={r['n']})" for r in plot_rows]
values = [signed_number(r["Edge"]) for r in plot_rows]
colors = ["#a33b35" if x < 0 else "#39755a" for x in values]
ax.barh(labels[::-1], values[::-1], color=colors[::-1])
ax.axvline(0, color="black", lw=.8)
ax.set_xlabel("Reported gross edge vs comparator (%)")
ax.set_title("F1 point estimates; interval endpoints unavailable")
fig.tight_layout()
plt.show()

## 10. Fixed Horizons

The published aggregate gives only the headline average and whether intervals exclude zero at one or more of three headline horizons; it does not give individual horizon estimates, Ns, or endpoint values. All nine predeclared horizons are displayed with explicit availability status below. The optional custom judge run returns all nine from the unchanged evaluator and preserves its original inference.

In [ ]:
horizon_status = pd.DataFrame({
    "horizon": FIXED_HORIZONS,
    "historical aggregate": ["Unavailable in committed summary"] * len(FIXED_HORIZONS),
    "custom judge": ["Available only when explicitly run"] * len(FIXED_HORIZONS),
})
display(horizon_status)
if JUDGE_RESULT is not None:
    display(pd.DataFrame(JUDGE_RESULT["tables"]["fresh_vs_ordinary"]))
# Availability matrix makes missing historical detail visible without inventing values.
fig, ax = plt.subplots(figsize=(9, 1.5))
ax.imshow([[0] * len(FIXED_HORIZONS)], cmap="Greys", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(FIXED_HORIZONS)), [str(h) for h in FIXED_HORIZONS])
ax.set_yticks([0], ["Historical aggregate"])
for i in range(len(FIXED_HORIZONS)):
    ax.text(i, 0, "NA", ha="center", va="center", fontsize=8)
ax.set_title("Fixed-horizon detail availability (NA = not reported)")
plt.show()

## 11. Ordinary-Day Baseline

The comparator uses sampled dates for the same tickers at least 30 days from any event, with 120 placebo draws per window in the historical study. The reported F1 edge is event return minus ordinary-day return. Absolute control means and full per-horizon matched sets are unavailable, so they remain blank in the static evidence. In the optional judge path the same original placebo and evaluator functions are called for the requested dates.

## 12. Sensitivity

All 36 reported cells are shown: 18 expiry/moneyness/entry settings per reported window. Every edge is negative in the committed table. The asterisks encode the source's count of headline horizons whose reported interval excluded zero; exact endpoint values are not present. This is a descriptive plateau, not a new strategy-selection step.

In [ ]:
SENSITIVITY = pd.DataFrame(sensitivity_rows(SUMMARY["runs/EXTRAS.md"]))
SENSITIVITY["otm_label"] = SENSITIVITY["otm"].map(lambda x: f"{x:.0%}")
display(SENSITIVITY.sort_values(["window", "bucket", "otm", "entry"]).reset_index(drop=True))
settings = [(b, o, e) for b in ("1m", "2m", "3-6m")
            for o in (0.03, 0.05, 0.10) for e in ("post", "pre")]
windows = ("in-sample", "out-of-sample")
heat = [[next(r["edge_pct"] for r in SENSITIVITY.to_dict("records")
              if r["window"] == w and r["bucket"] == b and r["otm"] == o and r["entry"] == e)
         for b, o, e in settings] for w in windows]
fig, ax = plt.subplots(figsize=(12, 2.8))
image = ax.imshow(heat, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
ax.set_yticks(range(2), windows)
ax.set_xticks(range(len(settings)), [f"{b} {o:.0%} {e}" for b,o,e in settings], rotation=55, ha="right")
for y,row in enumerate(heat):
    for x,value in enumerate(row): ax.text(x,y,f"{value:.2f}",ha="center",va="center",fontsize=7)
fig.colorbar(image, ax=ax, label="Fresh minus ordinary edge (%)")
ax.set_title("All 36 reported sensitivity cells")
fig.tight_layout()
plt.show()

## 13. Mechanism and Incremental Value

The committed F1 summary reports own-stock move edges of −1.32% in-sample and −1.36% in reported OOS. Median realized-to-implied move ratios were 1.08 versus 0.75 (fresh versus ordinary) in-sample and 0.82 versus 0.65 OOS; the OOS fresh median is below one. These are exploratory and do not establish causal mispricing. The challenge data lack index data, so market beta cannot be separated. The VRP H2 generalization test was near zero in-sample (+0.003, 95% CI −0.102 to +0.102, p=.999) and negative/not confirmed in reported OOS (−0.149, 95% CI −0.333 to +0.031, p=.114).

## 14. Costs, Liquidity, and Capacity

Costs are modeled from premium because quote data are unavailable. The 5% premium haircut per side corresponds to median 13 bps of notional per side in-sample (IQR 10–18) and 18 bps OOS (IQR 15–26); 2× cost doubles the assumed haircut. These medians cannot be subtracted from the mean edge to invent a net edge. Reported median option volume was 29 contracts per session in-sample and 18 OOS. The rough author-estimated capacity at 10% participation and ten concurrent positions was about $0.5M and $0.4M, respectively. The capacity estimate uses spot notional, not actual strike collateral, and is not guaranteed execution capacity.

In [ ]:
cost_capacity_tables = markdown_tables(SUMMARY["runs/EXTRAS.md"])
# Render committed cost/capacity source text as its own compact evidence block.
display(pd.DataFrame([
    {"Window":"in-sample", "median cost per side":"13 bps", "IQR":"10–18 bps", "median volume":"29 contracts/session", "rough capacity":"$0.5M"},
    {"Window":"reported OOS", "median cost per side":"18 bps", "IQR":"15–26 bps", "median volume":"18 contracts/session", "rough capacity":"$0.4M"},
]))
portfolio = pd.DataFrame([
    {"Window":"in-sample", "Costs":"1×", "n":55, "Annual return":"−1.2%", "Annual vol":"2.1%", "Sharpe":"−0.57", "Max DD":"−4.6%", "Worst event":"−16.5%"},
    {"Window":"in-sample", "Costs":"2×", "n":55, "Annual return":"−2.0%", "Annual vol":"2.1%", "Sharpe":"−0.97", "Max DD":"−5.1%", "Worst event":"−16.9%"},
    {"Window":"reported OOS", "Costs":"1×", "n":25, "Annual return":"−3.0%", "Annual vol":"2.3%", "Sharpe":"−1.31", "Max DD":"−2.5%", "Worst event":"−10.1%"},
    {"Window":"reported OOS", "Costs":"2×", "n":25, "Annual return":"−4.6%", "Annual vol":"2.3%", "Sharpe":"−1.98", "Max DD":"−3.5%", "Worst event":"−11.2%"},
])
display(portfolio)
print("Portfolio return is the separate 21-session fixed 1/10-capital simulation; it is not the per-event gross edge.")


## 15. Out-of-Sample

The F1 2026 OOS look was historically reported once: fresh versus ordinary edge −0.70%, n=25, with intervals including zero at all three headline horizons. Stale versus ordinary was +2.13% (n=40); fresh minus stale was −2.82% (n=25) and did not pass the in-sample significance gate. VRP pooled H2 had a separate reported OOS look (−0.149, not confirmed). No VRP category received OOS because none passed the in-sample BH gate. This notebook does not open or recompute OOS data. The actual judges' sealed date range and results are unknown; 2023-06..08 in the starter notebook is only a configured placeholder.

## 16. Historical Research Appendix

The committed appendix records the excerpt-date measurement failure; null or negative JEV leadership tests; sparse feasibility arms; B5 good-news long-call null; B6 earnings results that failed OOS; F1 fresh/stale results; and the VRP null map. The VRP strategy mapping is descriptive only and is not a validated trade. Its category `strategy edge` is gross versus ordinary days while `net P&L` is the strategy's own mean P&L after costs; do not treat them as equivalent. VRP categories overlap, the ordinary-day pool can fall back to a shared pool for sparse categories, and no H1 category received OOS. F1 also uses a static September 2026 top-100 universe, has no earnings calendar or index control, and its portfolio reindex/forward-fill can extend marks; none supports a blanket no-lookahead or causal claim. A documented missing-tickers data-shape failure in VRP counts was repaired upstream; this notebook does not rerun that stage. Experiments 8–12/9B, later blind reviews, numeric-guidance work, and implementation/source audits are absent from this published snapshot. The VRP missing-tickers repair is reported as resolved upstream, but was not independently rerun here; it is an implementation failure, not a market result.

The pinned four-page PDF mentions the planned variance-premium map but omits its completed results, which are present in the pinned Markdown. The publication bundle needs synchronization. The organizer starter says a two-page note limit, while the supplied task brief says five pages; this conflict is unresolved.

In [ ]:
VRP_TABLE = markdown_tables(SUMMARY["runs/APPENDIX_VRP.md"])[0]
display(pd.DataFrame(VRP_TABLE))

def interval_bounds(text):
    parts = re.findall(r"[+−-]?\d+\.\d+", text.replace("−", "-"))
    return float(parts[0]), float(parts[1])
vrp = [r for r in VRP_TABLE if r.get("category") and r["category"] != "H2_fresh_minus_stale"]
labels = [r["category"] for r in vrp]
point = [float(r["in-sample diff"].replace("−", "-")) for r in vrp]
ci = [interval_bounds(r["95% CI"]) for r in vrp]
fig, ax = plt.subplots(figsize=(8, 7))
y = list(range(len(vrp)))
ax.hlines(y, [x[0] for x in ci], [x[1] for x in ci], color="#66788a", lw=1.4)
ax.scatter(point, y, color="#315b7d", s=24, zorder=3)
ax.axvline(0, color="#a33b35", lw=1)
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlabel("Mean log(realized / implied move), event minus ordinary")
ax.set_title("VRP map: 18 in-sample estimates; all reported 95% CIs include zero")
fig.tight_layout()
plt.show()

## 17. Final Judge Summary

The pre-registered claim failed: fresh leadership 8-Ks did not make the short put outperform ordinary days. The negative sign repeated in the historically reported 2026 OOS look, but its interval included zero; fresh-minus-stale was not significant in-sample. The broader underpricing interpretation remains post hoc and unconfirmed. The 18-category VRP map is a null after multiple-testing correction, and its pooled H2 OOS result was not confirmed.

The table below is the authoritative static summary for the quant note. Detailed F1 horizon-specific values, numeric F1 CI endpoints, matched issuer counts, and absolute signal/control means are explicitly unavailable. Metrics provenance is pinned to the build commit and reconciled against Worker A's evidence packet; documented missing measurements remain unavailable.

In [ ]:
FINAL_SUMMARY = pd.DataFrame([
    {"Measure":"F1 fresh vs ordinary, IS", "Estimate":"−1.19%", "n":55, "95% CI":"excludes zero at 1/3 horizons", "Status":"Pre-registered claim rejected"},
    {"Measure":"F1 stale vs ordinary, IS", "Estimate":"−0.16%", "n":95, "95% CI":"includes zero", "Status":"Not supported"},
    {"Measure":"F1 fresh minus stale, IS", "Estimate":"−1.03%", "n":55, "95% CI":"includes zero at all 3", "Status":"Not supported"},
    {"Measure":"F1 fresh vs ordinary, reported 2026 OOS", "Estimate":"−0.70%", "n":25, "95% CI":"includes zero at all 3", "Status":"Same sign, not significant"},
    {"Measure":"VRP H2 fresh minus stale, IS", "Estimate":"+0.003", "n":"666 / 547", "95% CI":"−0.102 to +0.102", "Status":"p=.999; not supported"},
    {"Measure":"VRP H2 fresh minus stale, reported OOS", "Estimate":"−0.149", "n":"229 / 218", "95% CI":"−0.333 to +0.031", "Status":"p=.114; not confirmed"},
])
display(FINAL_SUMMARY)
metrics = load_metrics()
print("Metrics status:", metrics["reconciliation"]["status"])
print("Source commit:", metrics["build_commit"])
print("Packet reconciliation: complete against Worker A handoff; unavailable measurements remain null")